# E255 Task 1 — Physically Constrained Modeling and Surrogate Evaluation

**Student:** *(your name)* · **Dataset seed:** *(your assigned seed)* · **Date:** *(submission date)*

This notebook is your primary submission artifact. Run all cells before submitting. All outputs must be saved and visible. Your written responses replace the italicised placeholder text in each section.

**Provided materials (do not modify):** `pinn.npz`, `surrogate.npz`, `train.csv`, `val.csv`, `test.csv`, `reference_conditions.csv`, `case_physics.py`, `mlp.py`.

> **Download before closing:** this environment does not save automatically. Use File → Download to save your `.ipynb` after each working session.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Load course modules from the JupyterLite virtual filesystem
_ok = False
for _p in ['.', '/', '/drive', '/drive/v1/files']:
    try:
        if _p not in sys.path:
            sys.path.insert(0, _p)
        import case_physics, mlp
        _ok = True
        break
    except ImportError:
        continue

if not _ok:
    # Inline fallback
    import types
    case_physics = types.SimpleNamespace(
        CP_COOLANT=4186.0,
        energy_balance_Tout=lambda Q, mdot, Tin: np.asarray(Tin) + np.asarray(Q) / (np.asarray(mdot) * 4186.0)
    )
    def _load(path):
        z = np.load(path)
        n = sum(1 for k in z.files if k.startswith('W'))
        class _M:
            def __init__(self):
                self.W = [z[f'W{i}'] for i in range(n)]
                self.b = [z[f'b{i}'] for i in range(n)]
            def forward(self, X):
                h = X
                for i,(W,b) in enumerate(zip(self.W,self.b)):
                    z2 = h@W+b
                    h = np.tanh(z2) if i<len(self.W)-1 else z2
                return h
        return _M(), {k:z[k] for k in z.files if not (k[0] in 'Wb' and k[1:].isdigit())}
    mlp = types.SimpleNamespace(load=_load)

# Locate data files — drive is mounted at /drive/, notebook cwd is /drive/task1/
_cwd = os.getcwd()
_found = False
for _d in ['..', '/drive', '/', '/drive/v1/files', '.']:
    try:
        if os.path.exists(os.path.join(_d, 'train.csv')):
            os.chdir(_d)
            print(f'data dir: {os.path.abspath(_d)}')
            _found = True
            break
    except Exception:
        continue

if not _found:
    print(f'WARNING: train.csv not found. cwd was: {_cwd}')
    for _d in ['..', '/drive', '/', _cwd]:
        try:
            _ls = [f for f in os.listdir(_d) if not f.startswith('.')]
            print(f'  ls {_d!r}: {_ls[:10]}')
        except Exception as e:
            print(f'  ls {_d!r}: {e}')

CP = case_physics.CP_COOLANT
INPUTS = ['Q_W', 'mdot_kgs', 'Tin_C', 'w_mm', 'k_WmK']
pd.set_option('display.float_format', lambda v: f'{v:.3f}')

parts = {name: pd.read_csv(f'{name}.csv') for name in ('train', 'val', 'test')}
ref   = pd.read_csv('reference_conditions.csv')

pinn_model,  pinn_sc  = mlp.load('pinn.npz')
surr_model,  surr_sc  = mlp.load('surrogate.npz')

def predict(model, scalers, df):
    X = (df[INPUTS].values - scalers['mx']) / scalers['sx']
    return model.forward(X) * scalers['sy'] + scalers['my']

pinn_pred = {n: predict(pinn_model, pinn_sc, df) for n, df in parts.items()}
surr_pred = {n: predict(surr_model, surr_sc, df) for n, df in parts.items()}
print('Setup complete. Partitions:', {k: len(v) for k,v in parts.items()})

---
## Section A — Building and Validating the Physically Constrained Model

> **Rubric — Competent:** The submission demonstrates that the physically constrained model satisfies the required physical-consistency checks (energy-balance residual, boundary-condition residual, nonnegative thermal-property check, comparison with reference simulation values, physical-constraint satisfaction rate) across the training, validation, and held-out test partitions, with engineering interpretations of what each result means for model reliability.

### A1 — Energy-Balance Residual

The governing first-law energy balance: $T_{\text{out}} = T_{\text{in}} + Q / (\dot{m} \cdot c_p)$.
The residual is the model's predicted $T_{\text{out}}$ minus the first-law value.

In [ ]:
TOL_A1 = 1.0  # deg C — from the Surrogate Performance Log

def energy_residual(df, p):
    req = case_physics.energy_balance_Tout(df['Q_W'].values, df['mdot_kgs'].values, df['Tin_C'].values)
    return p[:, 0] - req

a1 = pd.DataFrame([
    {'partition': n,
     'mean |residual| (°C)': float(np.abs(energy_residual(df, pinn_pred[n])).mean()),
     'max |residual| (°C)':  float(np.abs(energy_residual(df, pinn_pred[n])).max()),
     'tolerance (°C)': TOL_A1,
     'result': 'PASS' if np.abs(energy_residual(df, pinn_pred[n])).mean() <= TOL_A1 else 'FAIL'}
    for n, df in parts.items()])
display(a1)

fig, ax = plt.subplots(figsize=(7, 3))
for n, df in parts.items():
    ax.hist(energy_residual(df, pinn_pred[n]), bins=40, alpha=0.55, label=n)
ax.axvline(0, color='k', lw=0.8)
ax.set_xlabel('Energy-balance residual (°C)'); ax.set_ylabel('Count')
ax.set_title('A1: Energy-balance residual distribution'); ax.legend()
plt.tight_layout(); plt.show()

*Student:* *(Replace this text. Interpret what the A1 result means for the model's engineering reliability — focus on the held-out test partition. What does a near-zero residual distribution tell you about whether the physics penalty worked?)*

### A3 — Nonnegative Thermal-Property Check

All predicted temperatures must be physically admissible — no sub-zero Celsius predictions inside the operating envelope.

In [ ]:
a3 = pd.DataFrame([
    {'partition': n,
     'negative Tout predictions': int((pinn_pred[n][:, 0] < 0).sum()),
     'negative Tmax predictions': int((pinn_pred[n][:, 1] < 0).sum()),
     'tolerance': 0,
     'result': 'PASS' if (pinn_pred[n] < 0).sum() == 0 else 'FAIL'}
    for n, df in parts.items()])
display(a3)

*Student:* *(Replace this text. Interpret the A3 result. Why is a nonnegative-temperature check more useful as an out-of-distribution guardrail than as an in-distribution metric?)*

### A4 — Comparison with Reference Simulation Values

The provided `reference_conditions.csv` gives five benchmark points (four edge conditions plus the envelope centre). Mean absolute error (MAE) against the reference characterises accuracy at conditions that span the operating range.

In [ ]:
TOL_A4 = 2.0  # deg C MAE — from the Surrogate Performance Log

rp = predict(pinn_model, pinn_sc, ref)
a4 = ref[['condition', 'Q_W', 'mdot_kgs', 'Tin_C']].copy()
a4['Tmax predicted (°C)'] = rp[:, 1]
a4['Tmax reference (°C)'] = ref['Tmax_ref_C']
a4['|error| (°C)']        = (a4['Tmax predicted (°C)'] - a4['Tmax reference (°C)']).abs()
mae = float(a4['|error| (°C)'].mean())
print(f"MAE vs reference: {mae:.3f} °C  (tolerance {TOL_A4}) → {'PASS' if mae <= TOL_A4 else 'FAIL'}")
display(a4)

*Student:* *(Replace this text. Which benchmark condition drives the error, and what does that say about the model's limitations? If A4 fails, is that a flaw in the model or a property of the training data — or both?)*

### Section A Summary

*Student:* *(Replace this text. Summarise the Section A results across all five checks. Which checks passed, which failed, and what does the overall pattern tell an engineer about whether this model is fit for use in the cold-plate design task?)*

---

## Section B — Explaining the Constraints

*(to be completed — placeholder)*

---

## Section C — Edge-Condition Viability

*(to be completed — placeholder)*

---

*Sections D–K follow the same pattern. Each opens with the Competent rubric descriptor, provides required computations, and ends with a student interpretation.*